[![Fixel Algorithms](https://fixelalgorithms.co/images/CCExt.png)](https://fixelalgorithms.gitlab.io)

# Deep Learning Methods

## Computer Vision - Object Detection - MNIST Object Detection

> Notebook by:
> - Royi Avital RoyiAvital@fixelalgorithms.com

## Revision History

| Version | Date       | User        |Content / Changes                                                   |
|---------|------------|-------------|--------------------------------------------------------------------|
| 1.0.000 | 12/10/2025 | Royi Avital | First version                                                      |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FixelAlgorithmsTeam/FixelCourses/blob/master/AIProgram/2024_02/0099DeepLearningObjectDetection.ipynb)

In [ ]:
# Import Packages

# General Tools
import numpy as np
import scipy as sp
import pandas as pd

# Machine Learning
from sklearn.model_selection import train_test_split

# Deep Learning
import torch
import torch.nn            as nn
import torch.nn.functional as F
import torchinfo

from torchmetrics.detection.mean_ap import MeanAveragePrecision

import torchvision
from torchvision.transforms import v2 as TorchVisionTrns

import torchvista

# Miscellaneous
import os
from platform import python_version
import random

# Typing
from typing import Callable, Dict, Generator, List, Optional, Self, Set, Tuple, Union
from numpy.typing import NDArray
from torch import Tensor

# Visualization
import matplotlib.pyplot as plt

# Jupyter
from IPython import get_ipython

## Notations

* <font color='red'>(**?**)</font> Question to answer interactively.
* <font color='blue'>(**!**)</font> Simple task to add code for the notebook.
* <font color='green'>(**@**)</font> Optional / Extra self practice.
* <font color='brown'>(**#**)</font> Note / Useful resource / Food for thought.

Code Notations:

```python
someVar    = 2; #<! Notation for a variable
vVector    = np.random.rand(4) #<! Notation for 1D array
mMatrix    = np.random.rand(4, 3) #<! Notation for 2D array
tTensor    = np.random.rand(4, 3, 2, 3) #<! Notation for nD array (Tensor)
tuTuple    = (1, 2, 3) #<! Notation for a tuple
lList      = [1, 2, 3] #<! Notation for a list
dDict      = {1: 3, 2: 2, 3: 1} #<! Notation for a dictionary
oObj       = MyClass() #<! Notation for an object
dfData     = pd.DataFrame() #<! Notation for a data frame
dsData     = pd.Series() #<! Notation for a series
hObj       = plt.Axes() #<! Notation for an object / handler / function handler
```

### Code Exercise

 - Single line fill

```python
valToFill = ???
```

 - Multi Line to Fill (At least one)

```python
# You need to start writing
?????
```

 - Section to Fill

```python
#===========================Fill This===========================#
# 1. Explanation about what to do.
# !! Remarks to follow / take under consideration.
mX = ???

?????
#===============================================================#
```

In [ ]:
# Configuration
# %matplotlib inline

seedNum = 512
np.random.seed(seedNum)
random.seed(seedNum)

# Matplotlib default color palette
lMatPltLibclr = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
# sns.set_theme() #>! Apply SeaBorn theme

runInGoogleColab = 'google.colab' in str(get_ipython())

# Improve performance by benchmarking
torch.backends.cudnn.benchmark = True

# Reproducibility (Per PyTorch Version on the same device)
# torch.manual_seed(seedNum)
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark     = False #<! Makes things slower

In [ ]:
# Constants

FIG_SIZE_DEF    = (8, 8)
ELM_SIZE_DEF    = 50
CLASS_COLOR     = ('b', 'r')
EDGE_COLOR      = 'k'
MARKER_SIZE_DEF = 10
LINE_WIDTH_DEF  = 2

PROJECT_NAME       = 'FixelCourses'
DATA_FOLDER_NAME   = 'DataSets'
MODELS_FOLDER_NAME = 'Models'
BASE_FOLDER_PATH   = os.getcwd()[:(len(os.getcwd()) - (os.getcwd()[::-1].lower().find(PROJECT_NAME.lower()[::-1])))]
DATA_FOLDER_PATH   = os.path.join(BASE_FOLDER_PATH, DATA_FOLDER_NAME)
MODELS_FOLDER_PATH = os.path.join(BASE_FOLDER_PATH, MODELS_FOLDER_NAME)

TENSOR_BOARD_BASE = 'TB'

In [ ]:
# Download Auxiliary Modules for Google Colab
if runInGoogleColab:
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataManipulation.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataVisualization.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DeepLearningPyTorch.py

In [ ]:
# Courses Packages

from DeepLearningObjectDetection import BBoxFormat, YoloDataset
from DeepLearningObjectDetection import CollateObjectDetection
from DataVisualization import PlotLabelsHistogram
from DeepLearningPyTorch import GetBatch, TrainModel

* <font color='blue'>(**!**)</font> Go through `GenLabeldDataEllipse()`.
* <font color='blue'>(**!**)</font> Go through `ObjectDetectionDataset`.
* <font color='blue'>(**!**)</font> Go through `YoloGrid`.

In [ ]:
# General Auxiliary Functions

def TensorImageNumpy( tZ: Tensor ) -> NDArray:
    """
    Converts a PyTorch Tensor to a Numpy Array.
    """
    mZ = tZ.squeeze()
    mX = mZ.detach().cpu().numpy()

    return mX

def TensorImgNumpy( tI: Tensor ) -> NDArray:
    """
    Converts a PyTorch Tensor Image to a Numpy Array Image.
    """
    
    mI = TensorImageNumpy(tI.permute(1, 2, 0))

    return mI

def PlotYoloImage( mI: NDArray, vY: Union[NDArray, Tensor], mB: Union[NDArray, Tensor], tuFigSize: Tuple[int, int] = (4, 6), lClrMap: List[str] = lMatPltLibclr, hA: plt.Axes = None ) -> None:
    """Plot an image with pixel XYXY boxes and digit labels."""

    if hA is None:
        hF, hA = plt.subplots(figsize = tuFigSize)
    else:
        hF = None
    hA.imshow(mI, cmap = 'gray')

    for valY, (xMin, yMin, xMax, yMax) in zip(vY, np.asarray(mB)):
        boxColor = lClrMap[int(valY)]
        hA.add_patch(plt.Rectangle((xMin, yMin), xMax - xMin, yMax - yMin,
                                  fill = False, edgecolor = boxColor, linewidth = LINE_WIDTH_DEF))
        hA.text(xMin, yMin, str(int(valY)), color = boxColor, fontsize = 14, va = 'bottom')

    hA.axis('off')
    if hF is not None:
        hF.tight_layout()

def PlotYoloBox( hA: plt.Axes, valY: Union[int, float], mB: Union[NDArray, Tensor], vS: Union[int, float], lClrMap: List[str] = lMatPltLibclr ) -> None:
    """Plot a single YOLO box on the given Axes."""

    xMin, yMin, xMax, yMax = np.asarray(mB)
    boxColor = lClrMap[int(valY)]
    hA.add_patch(plt.Rectangle((xMin, yMin), xMax - xMin, yMax - yMin,
                              fill = False, edgecolor = boxColor, linewidth = LINE_WIDTH_DEF))
    hA.text(xMin, yMin, f'{int(valY)} ({vS:.2f})', color = boxColor, fontsize = 14, va = 'bottom')

## Object Detection

This note presents real world like training of Object Detection model using the _MNIST Object Detection_.  

In [ ]:
# Parameters

# Data
datasetName = 'MNISTObjDet'
dataDirPath = os.path.join(DATA_FOLDER_PATH, datasetName) #<! Replace with the actual path to your dataset
boxFormat   = BBoxFormat.YOLO
tuImgSize   = (128, 128)

# Model
modelName   = 'ModelMNISTObjDet_2026_10_12.pt' #<! OneDrive -> Courses -> Models -> AIProgram
gridSize    = 5 #<! The grid is (gridSize x gridSize) 
imgMode     = torchvision.io.ImageReadMode.GRAY
numChannels = 1 if imgMode == torchvision.io.ImageReadMode.GRAY else 3

# Training

trainSampleRatio = 0.9 #<! Ratio of training images to total images
valSampleRatio   = 1 - trainSampleRatio #<! Ratio of validation images to total images

λObj = 1.0 #<! Objectness Loss weight
λReg = 1.0 #<! Localization Loss weight
λCls = 1.0 #<! Classification Loss weight
objPosWeight = 4.0 #<! Positive objectness weight relative to background cells
probThr = 0.5 #<! Probability Threshold (Score)
iouThr  = 0.5 #<! IoU Threshold (Score)

ϵ = 0.1 #<! Label Smoothing

batchSize   = 256
numWorkers  = 2 #<! Number of workers
numEpochs   = 35

# Visualization
numImg = 3

## Generate / Load Data

### MNIST Object Detection

A simple 10 class dataset for _Object Detection_ task.  
Each `(128, 128)` image contains MNIST digits. Their positions and scales are random.  
Annotation files use YOLO format. The `Dataset` returns boxes in pixel `XYXY` format.

* <font color='brown'>(**#**)</font> The pixel `XYXY` format is optimized for IoU calculations.

In [ ]:
# Generate Data

dsData = YoloDataset(dataDirPath, imgMode = imgMode)

numSamples = len(dsData)
numCls     = dsData.GetNumClasses()
print(f'The number of samples in the dataset: {numSamples}')
print(f'The number of classes in the dataset: {numCls}')

* <font color='brown'>(**#**)</font> One could use negative values for the bounding box. The model will extrapolate the object dimensions.

In [ ]:
# Image Sample

tXi, (vYi, mBi) = dsData[0]

print(f'The training data set data shape: {tXi.shape}')
print(f'The number of objects: {vYi.shape[0]}')
print(f'The training data set labels shape: {vYi.shape}')
print(f'The training data set box shape: {mBi.shape}')

### Plot the Data

In [ ]:
# Display Image Sample

tXi, (vYi, mBi) = dsData[0]
tXi = TensorImgNumpy(tXi)

PlotYoloImage(tXi, vYi, mBi)

In [ ]:
# Show Grid of Samples

hF, vHa = plt.subplots(3, 3, figsize = (8, 8))
vHa     = vHa.flat

for ii, hA in enumerate(vHa):
    sampleIdx = random.randint(0, len(dsData) - 1)
    tXi, (vYi, mBi) = dsData[sampleIdx]
    tXi = TensorImgNumpy(tXi)
    PlotYoloImage(tXi, vYi, mBi, hA = hA)

hF.tight_layout()

## Analysis of the Dataset

In [ ]:
# Histogram of Labels

lY, lB = dsData.GetTargets()
vY = np.concatenate(lY, axis = 0)
hA = PlotLabelsHistogram(vY);

In [ ]:
# Distribution of Box Centers
# Generate the 2D PDF of box centers

numBins = 32
mC = np.concatenate(lB, axis = 0)[:, :2]

hF, hA = plt.subplots(figsize = (6, 5))
mP, vXEdges, vYEdges, hImg = hA.hist2d(mC[:, 0], mC[:, 1], bins = numBins,
                                    range = [[0, 1], [0, 1]], density = True, cmap = 'viridis')
hA.set_title('PDF of Box Centers')
hA.set_xlabel('X Center (Normalized)')
hA.set_ylabel('Y Center (Normalized)')
hA.set_aspect('equal')
hA.invert_yaxis()
hF.colorbar(hImg, ax = hA, label = 'Probability Density')
hF.tight_layout()

In [ ]:
# Histogram of Box Size (Area)
# Shows the distribution of box sizes (Areas).
# Fitting a Gamma distribution to the box areas.

numBins = 32
mB = np.concatenate(lB, axis = 0)
vA = mB[:, 2] * mB[:, 3]

shapeVal, locVal, scaleVal = sp.stats.gamma.fit(vA, floc = 0)
vX = np.linspace(0, vA.max(), 512)[1:]
vP = sp.stats.gamma.pdf(vX, shapeVal, loc = locVal, scale = scaleVal)

hF, hA = plt.subplots(figsize = (6, 5))
hA.hist(vA, bins = numBins, density = True, edgecolor = 'k', alpha = 0.6, label = 'Box Areas')
hA.plot(vX, vP, color = 'r', linewidth = LINE_WIDTH_DEF,
        label = f'Gamma PDF (shape = {shapeVal:.2f}, scale = {scaleVal:.3f})')
hA.set_title('Histogram of Box Areas')
hA.set_xlabel('Box Area (Fraction of Image Area)')
hA.set_ylabel('Probability Density')
hA.legend()
hF.tight_layout()

### Analysis of the Grid Size

This section analyzes the effect of the grid size on the boundary of performance.

In [ ]:
# Analysis of Grid Size by Boxes
# Failure Rate at the image level.
# Failed image is an image with at least one cell containing 2 or more objects.

vGridSize = np.arange(3, 11)
lCenters = [np.asarray(mBoxes)[:, :2] for mBoxes in lB if len(mBoxes) > 1]
vDist = np.concatenate([sp.spatial.distance.pdist(mCenters) for mCenters in lCenters])
vCollision = np.zeros(len(vGridSize))

for gridIdx, gridSide in enumerate(vGridSize):
    numCases = 0
    for mCenters in lCenters:
        mCells = np.floor(mCenters * gridSide).astype(np.int64).clip(0, gridSide - 1)
        _, vCounts = np.unique(mCells, axis = 0, return_counts = True)
        numCases += np.any(vCounts > 1)
    vCollision[gridIdx] = numCases / len(lB)

vX = np.linspace(0, vDist.max(), 512)
hPdf = sp.stats.gaussian_kde(vDist)

hF, vHa = plt.subplots(1, 2, figsize = (12, 4))
vHa[0].plot(vX, hPdf(vX), linewidth = LINE_WIDTH_DEF)
vHa[0].set(title = 'Within Image Center Distances', xlabel = 'Normalized Euclidean Distance', ylabel = 'Probability Density')
vHa[1].plot(vGridSize, vCollision, marker = 'o', linewidth = LINE_WIDTH_DEF)
vHa[1].axhline(0.01, color = 'r', linestyle = '--', label = '1%')
vHa[1].set(title = 'At Least One Cell with 2+ Objects', xlabel = 'Grid Side S', ylabel = 'Collision Probability per Image')
vHa[1].set_xticks(vGridSize)
vHa[1].set_ylim(bottom = 0)
vHa[1].legend()
hF.tight_layout()

In [ ]:
# Analysis of Grid Size by Boxes
# Failure Rate at the object level.
# With one prediction per cell, n objects in a cell contribute n - 1 failures.


vGridSize = np.arange(3, 11)
numObjects = sum(len(mBoxes) for mBoxes in lB)
lCenters = [np.asarray(mBoxes)[:, :2] for mBoxes in lB if len(mBoxes) > 1]
vDist = np.concatenate([sp.spatial.distance.pdist(mCenters) for mCenters in lCenters])
vCollision = np.zeros(len(vGridSize))

for gridIdx, gridSide in enumerate(vGridSize):
    numCases = 0
    for mCenters in lCenters:
        mCells = np.floor(mCenters * gridSide).astype(np.int64).clip(0, gridSide - 1)
        _, vCounts = np.unique(mCells, axis = 0, return_counts = True)
        numCases += (vCounts - 1).sum()
    vCollision[gridIdx] = numCases / numObjects

vX = np.linspace(0, vDist.max(), 512)
hPdf = sp.stats.gaussian_kde(vDist)

hF, vHa = plt.subplots(1, 2, figsize = (12, 4))
vHa[0].plot(vX, hPdf(vX), linewidth = LINE_WIDTH_DEF)
vHa[0].set(title = 'Within Image Center Distances', xlabel = 'Normalized Euclidean Distance', ylabel = 'Probability Density')
vHa[1].plot(vGridSize, vCollision, marker = 'o', linewidth = LINE_WIDTH_DEF)
vHa[1].axhline(0.01, color = 'r', linestyle = '--', label = '1%')
vHa[1].set(title = 'Unrepresentable Objects', xlabel = 'Grid Side S', ylabel = 'Failure Rate over All Objects')
vHa[1].set_xticks(vGridSize)
vHa[1].set_ylim(bottom = 0)
vHa[1].legend()
hF.tight_layout()

* <font color='brown'>(**#**)</font> PyTorch with the `v2` transforms deals with bounding boxes using special type: `BoundingBoxes`.
* <font color='brown'>(**#**)</font> For _data augmentation_ see:
    - [Transforming and Augmenting Images](https://pytorch.org/vision/stable/transforms.html).
    - [Getting Started with Transforms v2](https://pytorch.org/vision/stable/auto_examples/transforms/plot_transforms_getting_started.html).
    - [Transforms v2: End to End Object Detection / Segmentation Example](https://pytorch.org/vision/stable/auto_examples/transforms/plot_transforms_e2e.html).
    - [How to Write Your Own v2 Transforms](https://pytorch.org/vision/stable/auto_examples/transforms/plot_custom_transforms.html).

### YOLO Detection Grid Tensor

The target data should be altered into a YOLO grid style.  
Namely the bounding box should be redefined relative to the grid cell the center of the box reside at.

![](https://i.imgur.com/CE1Ef7g.png)

This section implements the transformation as a PyTorch transformation.

In [ ]:
# Extract Data

tXi, (vYi, mBi) = dsData[10]
tXi = TensorImgNumpy(tXi)

In [ ]:
# Plot the Data
# Shows the centers over the grid set by `gridSize`

hF, hA = plt.subplots(figsize = (5, 5))
PlotYoloImage(tXi, vYi, mBi, hA = hA)

mB = np.asarray(mBi)
mC = (mB[:, :2] + mB[:, 2:]) / 2
numRows, numCols = tXi.shape[:2]

hA.vlines(np.linspace(0, numCols, gridSize + 1), 0, numRows, colors = 'w', linestyles = '--', alpha = 0.6)
hA.hlines(np.linspace(0, numRows, gridSize + 1), 0, numCols, colors = 'w', linestyles = '--', alpha = 0.6)
hA.scatter(mC[:, 0], mC[:, 1], c = 'r', marker = 'x', s = ELM_SIZE_DEF, linewidths = LINE_WIDTH_DEF, zorder = 3)
hA.set(xlim = (0, numCols), ylim = (numRows, 0))
hF.tight_layout()

### Data Loaders

This section defines the data loaded.  
It uses a special collate function which collate the different detections (Per image) into a list of dictionaries.  
A dictionary per image and the list for the whole batch.

In [ ]:
# Create Training and Validation Datasets

vIdxTrain, vIdxVal = train_test_split(np.arange(numSamples), test_size = valSampleRatio, train_size = trainSampleRatio, random_state = seedNum, shuffle = True)

dsTrain = torch.utils.data.Subset(dsData, vIdxTrain)
dsVal   = torch.utils.data.Subset(dsData, vIdxVal)

numSamplesTrain = len(dsTrain)
numSamplesVal   = len(dsVal)

print(f'The training data set contains  : {numSamplesTrain:4d} samples.')
print(f'The validation data set contains: {numSamplesVal:4d} samples.')

In [ ]:
# Data Loader

dlTrain = torch.utils.data.DataLoader(dsTrain, shuffle = True, batch_size = 1 * batchSize, num_workers = 0, collate_fn = CollateObjectDetection, drop_last = True, persistent_workers = False)
dlVal   = torch.utils.data.DataLoader(dsVal, shuffle = False, batch_size = 2 * batchSize, num_workers = 0, collate_fn = CollateObjectDetection, persistent_workers = False)

In [ ]:
# Iterate on the Loader
# The first batch.

tX, lY = GetBatch(dlTrain)

print(f'The batch features dimensions: {tX.shape}')
print(f'The targets length: {len(lY)}')
print(f'The dictionary keys per target: {lY[0].keys()}')

* <font color='brown'>(**#**)</font> The dimensions of the boxes and labels per sample might be different.  
Hence they can not be collated into a single tensor.  
The common solution is to pack them as a dictionary per sample.
* <font color='blue'>(**!**)</font> Go through `CollateObjectDetection()`.

## The Model

This section defines the model.  

* <font color='brown'>(**#**)</font> The following implementation has a model with a single output, both for the regression and the classification.
* <font color='brown'>(**#**)</font> One could create 2 different outputs (_Heads_) for each task.
* <font color='brown'>(**#**)</font> [Finally Understand Anchor Boxes in Object Detection (2D and 3D)](https://www.thinkautonomous.ai/blog/anchor-boxes/) - How the prior is set by clustering the data.
* <font color='brown'>(**#**)</font> Anchor Free Model: [Forget the Hassles of Anchor Boxes with FCOS: Fully Convolutional One Stage Object Detection](https://scribe.rip/fc0e25622e1c).

### Depth Wise Separable Convolution

The Depth Wise Separable Convolution merges 2 concept:
 - Spatial Convolutions in Groups.
 - Projection by `1x1` Convolution.

* <font color='brown'>(**#**)</font> See [Animated AI - Groups, Depthwise and Depthwise Separable Convolution (Neural Networks)](https://www.youtube.com/watch?v=vVaRhZXovbw).
* <font color='brown'>(**#**)</font> The concept was made popular by models such as EfficientNet ([EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks](https://arxiv.org/abs/1905.11946)) and [MobileNet](https://en.wikipedia.org/wiki/MobileNet).

In [ ]:
# Depth Wise Separable Convolutional Layer

class DepthWiseSeparableConv2D( nn.Module ):
    def __init__( self, inChannels: int, outChannels: int, kernelSize: int, stride: int = 1, padding: Union[int, str] = 0, bias: bool = True, *, dilation: int = 1 ) -> None:
        super(DepthWiseSeparableConv2D, self).__init__()

        self.oDepthWiseSeparableConv2D = nn.Sequential(
            nn.Conv2d(inChannels, inChannels, kernel_size = kernelSize, stride = stride, padding = padding, dilation = dilation, groups = inChannels, bias = bias), #<! Depthwise Convolution
            nn.Conv2d(inChannels, outChannels, kernel_size = 1, stride = 1, padding = 0, bias = bias)                                          #<! Pointwise Convolution
        )
    
    def forward( self, tX: Tensor ) -> Tensor:
        
        tX = self.oDepthWiseSeparableConv2D(tX)
        
        return tX


class ResidualDepthWiseBlock( nn.Module ):
    """Refine features with a cheap residual depthwise-separable convolution."""

    def __init__( self, numChannels: int, dilation: int = 1 ) -> None:
        super(ResidualDepthWiseBlock, self).__init__()

        # The depthwise convolution gathers spatial context independently in each channel.
        # Dilation increases the receptive field without reducing the feature-map resolution.
        self.oResidualBranch = nn.Sequential(
            nn.Conv2d(numChannels, numChannels, 3, padding = dilation, dilation = dilation, groups = numChannels, bias = False),
            nn.BatchNorm2d(numChannels),
            nn.ReLU(),
            # The pointwise convolution mixes information between channels.
            nn.Conv2d(numChannels, numChannels, 1, bias = False),
            nn.BatchNorm2d(numChannels),
        )
        self.oActivation = nn.ReLU()

    def forward( self, tX: Tensor ) -> Tensor:

        # The skip connection preserves the original representation and improves gradient flow.
        tX = self.oActivation(tX + self.oResidualBranch(tX))

        return tX

### The `µDetector` Architecture

The model uses its parameter budget to preserve spatial detail and build enough context for MNIST object detection.

 - **Backbone**: Three regular `3x3` convolutions expand the representation to `24`, `56`, and `112` channels.  
   Their stride of `2` reduces a `128x128` image to a detailed `16x16` feature map.
 - **Residual Context**: Three depthwise-separable residual blocks use dilation `1`, `2`, and `4`.  
   The residual paths improve gradient flow while the increasing dilation grows the receptive field without further downsampling.
 - **Neck**: A depthwise spatial convolution refines the feature map, then a pointwise convolution compresses it to `80` channels.  
   Adaptive average pooling is delayed until after this refinement and maps the features to the requested detection grid.
 - **Detection Heads**: Separate pointwise heads predict objectness and box coordinates versus class logits.  
   This avoids forcing localization and classification through the same final projection.
 - **Outputs**: Each cell predicts one objectness logit, `4` box values, and class logits.  
   Only the box values pass through `Sigmoid`: the center is relative to the cell, while width and height are relative to the image.

* <font color='brown'>(**#**)</font> Regular convolutions in the backbone provide more spatial and cross-channel capacity than a depthwise-only extractor.
* <font color='brown'>(**#**)</font> Each cell still predicts only one box. Two objects whose centers fall in the same cell cannot both be represented.

In [ ]:
# Model Class

class µDetector( nn.Module ):
    def __init__( self, numClasses: int, gridSize: int, /, *, numChannels: int = 3 ) -> None:
        super(µDetector, self).__init__()

        self.numClasses = numClasses
        self.gridSize   = gridSize

        # Regular convolutions provide rich spatial and cross-channel features.
        # Three stride-2 stages map a 128 x 128 image to a 16 x 16 feature map.
        self.oBackbone = nn.Sequential(
            nn.Conv2d(numChannels, 24, 3, stride = 2, padding = 1, bias = False), nn.BatchNorm2d(24), nn.ReLU(),
            nn.Conv2d(24, 56, 3, stride = 2, padding = 1, bias = False), nn.BatchNorm2d(56), nn.ReLU(),
            nn.Conv2d(56, 112, 3, stride = 2, padding = 1, bias = False), nn.BatchNorm2d(112), nn.ReLU(),
        )

        # Residual context blocks retain the 16 x 16 geometry. Increasing dilation
        # expands the receptive field to approximately the full input image.
        self.oContext = nn.Sequential(
            ResidualDepthWiseBlock(112, dilation = 1),
            ResidualDepthWiseBlock(112, dilation = 2),
            ResidualDepthWiseBlock(112, dilation = 4),
        )

        # The neck refines and compresses features before pooling them to the YOLO grid.
        self.oNeck = nn.Sequential(
            nn.Conv2d(112, 112, 3, padding = 1, groups = 112, bias = False), nn.BatchNorm2d(112), nn.ReLU(),
            nn.Conv2d(112, 80, 1, bias = False), nn.BatchNorm2d(80), nn.ReLU(),
            nn.AdaptiveAvgPool2d((gridSize, gridSize)),
        )

        # Separate projections let localization and classification specialize independently.
        self.oBoxHead = nn.Conv2d(80, 5, 1) #<! Objectness logit followed by 4 box values
        self.oClsHead = nn.Conv2d(80, numClasses, 1) #<! Class logits
    
    def forward( self, tX: Tensor ) -> Tensor:
        
        tX = self.oBackbone(tX)
        tX = self.oContext(tX)
        tX = self.oNeck(tX)

        tBox = self.oBoxHead(tX)
        tCls = self.oClsHead(tX)
        # Objectness and classes remain logits. Only box values are constrained to [0, 1].
        tBox = torch.cat((tBox[:, :1], torch.sigmoid(tBox[:, 1:])), dim = 1)
        tX = torch.cat((tBox, tCls), dim = 1)
        
        return tX

* <font color='brown'>(**#**)</font> Residual context blocks preserve the feature-map geometry while improving gradient flow.
* <font color='red'>(**?**)</font> Why are regular convolutions used in the downsampling backbone while depthwise-separable convolutions are used in the residual context blocks?
* <font color='brown'>(**#**)</font> Objectness and class predictions remain logits; only the four box values are normalized by `Sigmoid`.

In [ ]:
# Build the Model

oModel = µDetector(numCls, gridSize, numChannels = numChannels)

In [ ]:
# Model Information
# Pay attention to the layers name.
torchinfo.summary(oModel, tX.shape, col_names = ['kernel_size', 'input_size', 'output_size', 'num_params'], device = 'cpu', row_settings = ['depth', 'var_names'])

In [ ]:
# Model Visualization

torchvista.trace_model(oModel.eval(), tX)

* <font color='red'>(**?**)</font> Explain the dimensions of the last layer.
* <font color='red'>(**?**)</font> Will the model work with smaller images?

## Train the Model

This section trains the model.  

### Object Detection Loss

The loss combines weighted objectness classification, geometric box regression, and digit classification:

$$\ell = \lambda_{\text{Object}}\ell_{\text{Weighted BCE}} + \lambda_{\text{Regression}}\left(\ell_{\text{CIoU}} + 0.25\ell_{\text{Smooth L1}}\right) + \lambda_{\text{Classification}}\ell_{\text{CE}}.$$

Where:
 - $\ell_{\text{Weighted BCE}}$ classifies every grid cell as foreground or background. Positive cells receive `objPosWeight` times more weight because they are sparse relative to background cells.
 - $\ell_{\text{CIoU}}$ directly optimizes box overlap, center distance, and aspect-ratio agreement after decoding positive-cell predictions to normalized image coordinates.
 - $\ell_{\text{Smooth L1}}$ stabilizes learning of the encoded cell-relative center and image-relative width and height.
 - $\ell_{\text{CE}}$ classifies the digit only in cells containing a labeled object.
 - $\lambda_{\text{Object}}$, $\lambda_{\text{Regression}}$, and $\lambda_{\text{Classification}}$ control the contribution of each task.

* <font color='brown'>(**#**)</font> The objectness weight changes the relative cost of missed objects without weakening the penalty for false-positive background cells.
* <font color='brown'>(**#**)</font> The box target uses cell-relative `(cx, cy)` and image-relative `(w, h)`, but CIoU is calculated after both prediction and target are decoded to normalized image `XYXY` boxes.

### CIoU Loss

The _Complete Intersection over Union_ (CIoU) compares two boxes using three geometric properties:

1. **Overlap**: Rewards a large intersection between the predicted and target boxes.
2. **Center Distance**: Penalizes boxes whose centers are far apart, even when they do not overlap.
3. **Shape Agreement**: Penalizes differences in width to height ratio (Aspect ratio).

$$\ell_{\text{CIoU}} = 1 - \operatorname{IoU} + \frac{\rho^2\left(\boldsymbol{c}_{\text{pred}}, \boldsymbol{c}_{\text{target}}\right)}{d^2} + \alpha v.$$

Here, $\rho$ is the distance between box centers, $d$ is the diagonal of the smallest box enclosing both boxes, and $\alpha v$ measures aspect ratio disagreement.  
Intuitively, _CIoU_ asks the predicted box to **overlap the target, move its center toward it, and match its shape**.

### Weighted Binary Cross Entropy Loss

Each grid cell predicts whether it contains an object.  
Since most cells contain only background, ordinary binary cross entropy can let the many easy negative cells dominate the loss.

Weighted binary cross entropy gives positive cells an additional weight $p$:

$$\ell_{\text{WBCE}} = -\left[p y\log\left(\sigma(z)\right) + (1-y)\log\left(1-\sigma(z)\right)\right].$$

Here, $z$ is the objectness logit, $y \in \{0,1\}$ is the target, and $p$ is `objPosWeight`.  
With `objPosWeight = 4`, missing an object contributes four times the positive cell loss, while the loss for background cells remains unchanged.  
Intuitively, the model is told: **objects are rare, so pay more attention when one is present**.

In [ ]:
# Object Detection Loss
class ObjDetLoss( nn.Module ):
    def __init__( self, tuImgSize: Tuple[int, int], numCls: int, λObj: float, λReg: float, λCls: float, objPosWeight: float = 4.0, ϵ: float = 0.0 ) -> None:
        super(ObjDetLoss, self).__init__()

        self.numCls      = numCls
        self.λObj        = λObj #<! Objectness Loss Weight
        self.λReg        = λReg #<! Regression Loss Weight
        self.λCls        = λCls #<! Classification Loss Weight
        self.ϵ           = ϵ
        self.tuImgSize   = tuImgSize
        self.boxL1Weight = 0.25
        # Positive objectness cells are sparse. pos_weight increases their BCE contribution
        # without reducing the penalty for false positives in background cells.
        self.oObjLoss = nn.BCEWithLogitsLoss(pos_weight = torch.tensor(objPosWeight)) #<! `BCEWithLogitsLoss` requires `pos_weight` to be a tensor
        # Smooth L1 stabilizes encoded center/size learning while CIoU below directly
        # optimizes overlap, center distance and aspect-ratio agreement.
        self.oRegLoss = nn.SmoothL1Loss()
        self.oClsLoss = nn.CrossEntropyLoss(label_smoothing = ϵ)
    
    def forward( self: Self, tYHat: torch.Tensor, lY: List[Dict[str, Tensor]] ) -> torch.Tensor:
        """Compute weighted objectness, CIoU localization and classification losses.

        Input:
            tYHat: Tensor of shape (batchSize, 5 + numCls, gridSize, gridSize).
                Each cell contains [Objectness, cx, cy, w, h, Class1, ..., ClassN].
                Objectness and class values are logits.
                (cx, cy) are normalized within the cell.
                (w, h) are normalized to the whole image.
            lY: List of length batchSize. Each image has a target dictionary:
                'Boxes': Tensor of shape (numObjects, 4) in pixel XYXY format.
                'Labels': Tensor of shape (numObjects,) with zero-based class indices.

        Output:
            Scalar weighted sum of objectness, localization and classification losses.

        Remarks:
            Positive objectness cells are weighted by objPosWeight.
            Localization combines CIoU on decoded boxes with 0.25 * Smooth L1 on encoded boxes.
            tuImgSize is (imageHeight, imageWidth) and must match the target canvas.
        """

        batchSize = tYHat.shape[0]
        gridSize  = tYHat.shape[2]
        runDevice = tYHat.device
        numRows, numCols = self.tuImgSize

        # Build targets in YOLO grid format (Per cell)
        tObjTgt = torch.zeros((batchSize, gridSize, gridSize), device = runDevice, dtype = tYHat.dtype)
        tRegTgt = torch.zeros((batchSize, gridSize, gridSize, 4), device = runDevice, dtype = tYHat.dtype)
        tClsTgt = torch.zeros((batchSize, gridSize, gridSize), device = runDevice, dtype = torch.long)

        for ii in range(batchSize):
            tB = lY[ii]['Boxes'].to(device = runDevice, dtype = tYHat.dtype)
            tC = lY[ii]['Labels'].to(device = runDevice, dtype = torch.long)
            
            if tB.numel() == 0:
                continue
            
            tB = torchvision.ops.box_convert(tB, in_fmt = 'xyxy', out_fmt = 'cxcywh')
            tB = tB / tB.new_tensor([numCols, numRows, numCols, numRows]) #<! Normalize the coordinates (YOLO style)

            # Convert to grid coordinates (Same logic as `YoloGrid`)
            vCxNorm, vCyNorm, vW, vH = tB.T
            vCellX = (vCxNorm * gridSize).floor().to(torch.long).clamp_(0, gridSize - 1)
            vCellY = (vCyNorm * gridSize).floor().to(torch.long).clamp_(0, gridSize - 1)
            vCxRel = vCxNorm * gridSize - vCellX.to(vCxNorm.dtype)
            vCyRel = vCyNorm * gridSize - vCellY.to(vCyNorm.dtype)
            
            tObjTgt[ii, vCellY, vCellX]    = 1.0
            tRegTgt[ii, vCellY, vCellX, :] = torch.stack((vCxRel, vCyRel, vW, vH), dim = 1)
            tClsTgt[ii, vCellY, vCellX]    = tC.to(torch.long)
        
        # Weighted BCE is evaluated over foreground and background grid cells.
        objLoss = self.oObjLoss(tYHat[:, 0, :, :], tObjTgt)
        
        # Regression and classification are evaluated only in cells containing GT objects.
        mObjMask = tObjTgt.to(torch.bool)
        if mObjMask.any():
            tYHatReg = tYHat[:, 1:5, :, :].permute(0, 2, 3, 1) #<! (B, G, G, 4)
            mBPred = tYHatReg[mObjMask]
            mBTgt  = tRegTgt[mObjMask]

            # Decode positive cells to normalized image XYXY boxes before calculating CIoU.
            # This makes the main regression term match the evaluation geometry.
            _, vCellY, vCellX = torch.where(mObjMask)
            mBPredImg = mBPred.clone()
            mBTgtImg  = mBTgt.clone()
            mBPredImg[:, 0] = (vCellX + mBPredImg[:, 0]) / gridSize
            mBPredImg[:, 1] = (vCellY + mBPredImg[:, 1]) / gridSize
            mBTgtImg[:, 0]  = (vCellX + mBTgtImg[:, 0]) / gridSize
            mBTgtImg[:, 1]  = (vCellY + mBTgtImg[:, 1]) / gridSize
            mBPredImg = torchvision.ops.box_convert(mBPredImg, in_fmt = 'cxcywh', out_fmt = 'xyxy')
            mBTgtImg  = torchvision.ops.box_convert(mBTgtImg, in_fmt = 'cxcywh', out_fmt = 'xyxy')

            ciouLoss = torchvision.ops.complete_box_iou_loss(mBPredImg, mBTgtImg, reduction = 'mean') #<! CIoU loss
            regLoss  = ciouLoss + (self.boxL1Weight * self.oRegLoss(mBPred, mBTgt))

            tYHatCls = tYHat[:, 5:, :, :].permute(0, 2, 3, 1) #<! (B, G, G, numCls)
            clsLoss  = self.oClsLoss(tYHatCls[mObjMask], tClsTgt[mObjMask])
        else:
            regLoss = tYHat.new_zeros(())
            clsLoss = tYHat.new_zeros(())

        lossVal = (self.λObj * objLoss) + (self.λReg * regLoss) + (self.λCls * clsLoss)
        
        return lossVal

* <font color='brown'>(**#**)</font> CIoU supplies direct geometric supervision, while the smaller Smooth L1 term stabilizes the encoded center and size values early in training.
* <font color='green'>(**@**)</font> Compare CIoU with GIoU or DIoU while keeping all other training parameters fixed.

### Object Detection Score

This section implements the **mean per-image F1** score.

A _one to one_ assignment separates predictions and labeled objects into matched pairs, unmatched predictions and unmatched labeled objects:

 - **TP**: When prediction match the labeled detection with IoU above the threshold, score (Confidence Level) above the threshold and the same class.
 - **FP**: A prediction above the confidence threshold that remains unmatched after one-to-one assignment. This includes duplicate detections.
 - **FN**: A labeled object with no valid matched prediction. This is a missed detection.

The calculation of the score uses _one to one_ assignment to maximize valid matches. Each object can be matched once.
The score, per image, based on counting each type:

$$F_1 = \frac{2TP}{2TP + FP + FN}.$$

If both predictions and GT are empty, define ${F}_{1}$ as `1`.

* <font color='brown'>(**#**)</font> The per image ${F}_{1}$ is calculated independently for each image. Batch means are averaged using batch size weights.  
  Naive training loop supports this. The _mAP_ instead combines predictions across the full validation set. It requires a final calculation after the last validation batch.

In [ ]:
# Object Detection Score
class ObjDetScore( nn.Module ):
    def __init__( self, tuImgSize: Tuple[int, int], numCls: int, gridSize: int, probThr: float, iouThr: float, runDevice: torch.device ) -> None:
        super(ObjDetScore, self).__init__()

        self.numCls    = numCls
        self.gridSize  = gridSize
        self.probThr   = probThr
        self.iouThr    = iouThr
        self.runDevice = runDevice
        self.tuImgSize = tuImgSize
    
    @torch.inference_mode()
    def forward( self: Self, tYHat: torch.Tensor, lY: List[Dict[str, Tensor]] ) -> float:
        """Compute mean per-image F1 using one to one class and IoU matching.

        Input:
            tYHat: Tensor of shape (batchSize, 5 + numCls, gridSize, gridSize).
                Each cell contains [Objectness, cx, cy, w, h, Class1, ..., ClassN].
                Objectness and class values are logits.
                (cx, cy) are normalized within the cell.
                (w, h) are normalized to the whole image.
            lY: List of length batchSize. Each image has a target dictionary:
                'Boxes': Tensor of shape (numObjects, 4) in pixel XYXY format
                    [xMin, yMin, xMax, yMax], not normalized coordinates.
                'Labels': Tensor of shape (numObjects,) with zero-based class indices.
                The number of objects may differ between images.

        Output:
            Scalar mean F1 in [0, 1] across the batch. Higher is better.
            An image with no GT and no retained predictions scores 1.
            An empty batch returns 0.

        Remarks:
            tuImgSize is (imageHeight, imageWidth) and must match the target canvas.
            Predictions require sigmoid(Objectness) >= probThr.
            Matches require equal classes and IoU >= iouThr.
        """

        tYHat = tYHat.detach().float().cpu()
        numRows, numCols = self.tuImgSize
        valScore = 0.0

        for sampleIdx, dTarget in enumerate(lY):
            mBGt = dTarget['Boxes'].detach().float().cpu()
            vYGt = dTarget['Labels'].detach().cpu()
            vPy, vPx = torch.where(torch.sigmoid(tYHat[sampleIdx, 0]) >= self.probThr)
            numGt, numPred = len(vYGt), len(vPx)

            if numGt + numPred == 0:
                valScore += 1.0
                continue
            if numGt == 0 or numPred == 0:
                continue

            mBPred = tYHat[sampleIdx, 1:5, vPy, vPx].T.clone()
            mBPred[:, :2] = (mBPred[:, :2] + torch.stack((vPx, vPy), dim = 1)) / self.gridSize
            mBPred = torchvision.ops.box_convert(mBPred, in_fmt = 'cxcywh', out_fmt = 'xyxy')
            mBPred *= mBPred.new_tensor([numCols, numRows, numCols, numRows])
            vYPred = tYHat[sampleIdx, 5:5 + self.numCls, vPy, vPx].argmax(dim = 0)

            mIoU = torchvision.ops.box_iou(mBPred, mBGt)
            mMatch = ((mIoU >= self.iouThr) & (vYPred[:, None] == vYGt[None, :])).numpy()
            vPredIdx, vGtIdx = sp.optimize.linear_sum_assignment(mMatch, maximize = True)
            numTp = int(mMatch[vPredIdx, vGtIdx].sum())
            valScore += 2 * numTp / (numPred + numGt)

        return float(valScore / len(lY)) if lY else 0.0

* <font color='brown'>(**#**)</font> Both `ObjDetLoss` and `ObjDetScore` are not optimized and generates a bottleneck during training.

In [ ]:
# Run Device

runDevice = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu') #<! The 1st CUDA device
print(f'Run Device: {runDevice}')

In [ ]:
# Loss and Score Function

hL = ObjDetLoss(tuImgSize, numCls = numCls, λObj = λObj, λReg = λReg, λCls = λCls, objPosWeight = objPosWeight, ϵ = ϵ)
hS = ObjDetScore(tuImgSize, numCls = numCls, gridSize = gridSize, probThr = probThr, iouThr = iouThr, runDevice = runDevice)

hL = hL.to(runDevice)
hS = hS.to(runDevice)

In [ ]:
# Training Loop

oModel = oModel.to(runDevice)
oOpt = torch.optim.AdamW(oModel.parameters(), lr = 1e-5, betas = (0.9, 0.99), weight_decay = 1e-5) #<! Define optimizer
oSch = torch.optim.lr_scheduler.OneCycleLR(oOpt, max_lr = 7.5e-4, total_steps = numEpochs)
_, lTrainLoss, lTrainScore, lValLoss, lValScore, lLearnRate = TrainModel(oModel, dlTrain, dlVal, oOpt, numEpochs, hL, hS, oSch = oSch)

In [ ]:
# Load the Model and Training History

modelPath = os.path.join(MODELS_FOLDER_PATH, modelName)
if os.path.isfile(modelPath):
    dModel = torch.load(modelPath, map_location = runDevice, weights_only = True)
    dModelParams = dModel['ModelParams']
    dTrainParams = dModel['TrainParams']

    numCls       = dModelParams['numClasses']
    gridSize     = dModelParams['gridSize']
    tuImgSize    = tuple(dTrainParams['tuImgSize'])
    λObj          = dTrainParams['weightObj']
    λReg          = dTrainParams['weightReg']
    λCls          = dTrainParams['weightCls']
    objPosWeight = dTrainParams['objPosWeight']
    ϵ = dTrainParams['labelSmoothing']
    probThr = dTrainParams['probThr']
    iouThr  = dTrainParams['iouThr']

    oModel = µDetector(numCls, gridSize, numChannels = dModelParams['numChannels']).to(runDevice)
    oModel.load_state_dict(dModel['Model'])
    hL = ObjDetLoss(tuImgSize, numCls = numCls, λObj = λObj, λReg = λReg, λCls = λCls, objPosWeight = objPosWeight, ϵ = ϵ).to(runDevice)
    hS = ObjDetScore(tuImgSize, numCls = numCls, gridSize = gridSize, probThr = probThr, iouThr = iouThr, runDevice = runDevice).to(runDevice)

    lTrainLoss  = dModel['lTrainLoss']
    lTrainScore = dModel['lTrainScore']
    lValLoss    = dModel['lValLoss']
    lValScore   = dModel['lValScore']
    lLearnRate  = dModel['lLearnRate']
    print(f'Model and training history loaded from: {modelPath}')

oModel = oModel.eval()

In [ ]:
# Plot Training Phase

hF, vHa = plt.subplots(nrows = 1, ncols = 3, figsize = (12, 5))
vHa = np.ravel(vHa)

hA = vHa[0]
hA.plot(lTrainLoss, lw = 2, label = 'Train')
hA.plot(lValLoss, lw = 2, label = 'Validation')
hA.set_title(f'Object Detection Loss (λ = {λReg:0.1f})')
hA.set_xlabel('Epoch')
hA.set_ylabel('Loss')
hA.legend()

hA = vHa[1]
hA.plot(lTrainScore, lw = 2, label = 'Train')
hA.plot(lValScore, lw = 2, label = 'Validation')
hA.set_title('Object Detection Score')
hA.set_xlabel('Epoch')
hA.set_ylabel('Score')
hA.legend()

hA = vHa[2]
hA.plot(lLearnRate, lw = 2)
hA.set_title('Learn Rate Scheduler')
hA.set_xlabel('Epoch')
hA.set_ylabel('Learn Rate');

In [ ]:
# Plot Prediction

rndIdx = np.random.randint(numSamplesVal)

tX, tuY = dsVal[rndIdx]
vYGt    = tuY[0] #<! GT labels (D,)
mBGt    = tuY[1] #<! GT boxes (D, 4) in pixel XYXY format

with torch.inference_mode():
    tX = torch.as_tensor(tX)
    tX = torch.unsqueeze(tX, 0)
    tX = tX.to(device = runDevice)
    tYHat = oModel(tX).detach().cpu()

# Extract predictions with objectness above threshold
tYHat = tYHat[0] #<! (5 + numCls, gridSize, gridSize)

# Apply sigmoid to objectness (RAW logits)
mObjProb  = torch.sigmoid(tYHat[0, :, :]) #<! (G, G)
# Apply threshold
mPredMask = mObjProb.gt(probThr)          #<! (G, G)

# Get indices of cells with predictions
mIdx     = torch.nonzero(mPredMask, as_tuple = False) #<! (P, 2) -> (y, x)
numPred  = mIdx.shape[0]

if numPred > 0:
    vPy, vPx = mIdx[:, 0], mIdx[:, 1]
    
    # Extract predicted boxes: cell-relative CXCYWH -> pixel XYXY
    mBPred = tYHat[1:5, vPy, vPx].T #<! (P, 4) -> (cx_cell, cy_cell, w, h)
    mBPred = mBPred.clone()
    mBPred[:, 0] = (vPx.to(mBPred.dtype) + mBPred[:, 0]) / gridSize #<! cx normalized
    mBPred[:, 1] = (vPy.to(mBPred.dtype) + mBPred[:, 1]) / gridSize #<! cy normalized
    mBPred = torchvision.ops.box_convert(mBPred, in_fmt = 'cxcywh', out_fmt = 'xyxy')
    mBPred *= mBPred.new_tensor([tuImgSize[1], tuImgSize[0], tuImgSize[1], tuImgSize[0]])
    
    # Extract predicted classes
    tClsPred = tYHat[5:, vPy, vPx]                     #<! (numCls, P)
    vClsPred = torch.argmax(tClsPred, dim = 0).numpy() #<! (P,)
    
    # Extract objectness scores for the predictions
    vObjScore = mObjProb[vPy, vPx].numpy() #<! (P,)
    
    mBPred = mBPred.numpy() #<! (P, 4)
else:
    mBPred    = np.empty((0, 4))
    vClsPred  = np.array([])
    vObjScore = np.array([])

print(f'Number of GT objects: {len(vYGt)}')
print(f'Number of predictions above threshold: {numPred}')
print(f'Predicted boxes shape: {mBPred.shape}')
print(f'GT boxes shape: {mBGt.shape}')

# Plot GT and Predictions
mImg = TensorImgNumpy(tX[0])

hF, hA = plt.subplots(figsize = FIG_SIZE_DEF)
PlotYoloImage(mImg, vYGt, mBGt, hA = hA) #<! Plot GT
for ii, vB in enumerate(mBPred):
    PlotYoloBox(hA, vClsPred[ii], vB, vObjScore[ii]) #<! Plot Predictions

### Object Detection Score: _mAP_

**Mean Average Precision (mAP)** evaluates detections across the dataset.

 - Rank predictions by confidence within each class.
 - Match predictions to GT objects in the same image using an IoU threshold. Each object can be matched once. Duplicate detections are false positives.
 - **Precision** is the fraction of predictions that are correct. **Recall** is the fraction of GT objects found.
 - **AP** is the area under the interpolated precision-recall curve. **mAP** averages AP across classes.

* <font color='brown'>(**#**)</font> Calculation of the _mAP_ requires iterating over the whole validation set.  
  Hence in this use case it used only post training.
* <font color='brown'>(**#**)</font> The common notation `mAP@0.5` uses an IoU threshold of `0.5`.  
  Similarly, `mAP@0.5:0.95` averages over thresholds from `0.5` to `0.95` in steps of `0.05`.

In [ ]:
# Calculate the mAP over the Validation Set
# Using TorchMetrics' `MeanAveragePrecision`.

lIoU = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
oMap = MeanAveragePrecision(box_format = 'xyxy', iou_type = 'bbox',
                            iou_thresholds = lIoU, extended_summary = True,
                            backend = 'faster_coco_eval')

oModel.eval()
with torch.inference_mode():
    for tX, lY in dlVal:
        numRows, numCols = tX.shape[-2:]
        tYHat = oModel(tX.to(runDevice)).float().cpu()
        numGridRows, numGridCols = tYHat.shape[-2:]
        mCellY, mCellX = torch.meshgrid(torch.arange(numGridRows), torch.arange(numGridCols), indexing = 'ij')
        mCells = torch.stack((mCellX.flatten(), mCellY.flatten()), dim = 1)
        lPred, lTarget = [], []

        for tPred, dTarget in zip(tYHat, lY):
            mBPred = tPred[1:5].permute(1, 2, 0).reshape(-1, 4).clone()
            mBPred[:, :2] = (mBPred[:, :2] + mCells) / mBPred.new_tensor([numGridCols, numGridRows])
            mBPred = torchvision.ops.box_convert(mBPred, in_fmt = 'cxcywh', out_fmt = 'xyxy')
            mBPred *= mBPred.new_tensor([numCols, numRows, numCols, numRows])
            vYPred = tPred[5:].flatten(1).argmax(dim = 0)
            vScores = torch.sigmoid(tPred[0]).flatten()

            lPred.append({'boxes': mBPred, 'scores': vScores, 'labels': vYPred})
            lTarget.append({'boxes': dTarget['Boxes'].detach().float().cpu(),
                            'labels': dTarget['Labels'].detach().long().cpu()})

        oMap.update(lPred, lTarget)

dMap = oMap.compute()

tPrecision = dMap['precision'][:, :, :, 0, -1]
vMap = torch.stack([mPrecision[mPrecision >= 0].mean() for mPrecision in tPrecision]).numpy()

hF, hA = plt.subplots(figsize = (6, 4))
hA.plot(lIoU, vMap, marker = 'o', linewidth = LINE_WIDTH_DEF)
hA.set(title = f'Validation mAP by IoU, mAP@0.50:0.95: {dMap["map"].item():.3f}', xlabel = 'IoU Threshold', ylabel = 'mAP', ylim = (0, 1))
hA.set_xticks(lIoU)
hA.grid(alpha = 0.3)
hF.tight_layout()